# Getting started with GEDI L2A data in Python

This notebook introduces the **GEDI Level 2A Elevation and Height Metrics** product
([GEDI02_A.002](https://doi.org/10.5067/GEDI/GEDI02_A.002)). You will learn how to

1. download GEDI data and find your way around the HDF5 file structure,
2. turn GEDI laser shots into a GeoDataFrame and clip them to a study area,
3. read and plot the **relative height (RH) metrics** of a single shot,
4. quality-filter the data and plot a beam transect,
5. map canopy height and export the result for use in a GIS.

**How to use this notebook:** run the cells from top to bottom with `Shift + Enter`.
Nothing needs to be installed. On Binder your session is temporary. If you want to keep your work,
use *File → Download* before you close the tab.

> Adapted from the LP DAAC tutorial *Getting Started with GEDI L2A Version 2 Data in Python*
> ([nasa/GEDI-Data-Resources](https://github.com/nasa/GEDI-Data-Resources), Apache License 2.0).
> See `NOTICE` in the repository for details.

## 1. Setup

In [ ]:
# Load the Python packages we need
from pathlib import Path
import urllib.request
import zipfile

import h5py                  # read HDF5 files (the GEDI file format)
import numpy as np           # arrays and maths
import pandas as pd          # tables
import geopandas as gpd      # tables with geometries (points, polygons)
import matplotlib.pyplot as plt

DATA = Path("../data")       # folder with the GEDI files
OUT = Path("../output")      # folder for our results
OUT.mkdir(exist_ok=True)

## 1.1 Download the data

GEDI data are distributed by NASA's LP DAAC. A full GEDI file (*granule*) covers a quarter of an orbit
around the Earth and is 1–3 GB. Downloading from NASA requires a free Earthdata account.

Your instructor has already downloaded GEDI files covering the Kalkalpen region and put them
on a file share, so you can download them without an account. This can take a few minutes,
so start the cell below right away:

In [ ]:
# Download link provided by your instructor (a .zip with .h5 files)
DATA_URLS = [
    "https://share.uni-graz.at/download.php?token=578f94d7-b97a-4492-9a36-4174ca6df5d3&files_ids=132099",
]

for url in DATA_URLS:
    zip_path = DATA / "gedi_data.zip"
    print("Downloading ... (this can take a few minutes)")
    urllib.request.urlretrieve(url, zip_path)

    print("Unpacking ...")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if name.endswith(".h5"):
                (DATA / Path(name).name).write_bytes(z.read(name))
    zip_path.unlink()        # delete the zip, we only need the .h5 files
    print("Done.")

For reference, this is how the original files are downloaded from NASA with the `earthaccess` package
(needs an Earthdata login, so **don't run this here**):

```python
import earthaccess
earthaccess.login()
results = earthaccess.search_data(
    short_name="GEDI02_A", version="002",
    bounding_box=(14.25, 47.68, 14.62, 47.88),         # west, south, east, north
    temporal=("2022-05-01", "2023-09-30"))
earthaccess.download(results[:1], local_path="raw/")  # ~1-3 GB per granule!
```

In [ ]:
# List all GEDI files in the data folder
files = sorted(DATA.glob("*GEDI02_A*.h5"))

if len(files) == 0:
    print("No GEDI files found - run the download cell above first.")

for i, f in enumerate(files):
    size_mb = f.stat().st_size / 1e6
    print(i, f.name, f"({size_mb:.1f} MB)")

GEDI file names follow a fixed pattern, e.g. `GEDI02_A_2019170155833_O02932_02_T02267_02_003_01_V002.h5`:

| Part | Meaning |
|---|---|
| `GEDI02_A` | product short name |
| `2019170155833` | acquisition date and time (YYYY DDD HHMMSS, DDD = day of year) |
| `O02932` | orbit number |
| `02` | sub-orbit granule (1–4) |
| `T02267` | track number |
| `02` | positioning/pointing type (02 or higher = final) |
| `003` / `01` | processing software version / granule production version |
| `V002` | product version |

**Question:** On which date was the first file acquired?

## 2. Explore the HDF5 file

HDF5 files are organised like a file system: **groups** (folders) contain **datasets** (arrays) and **attributes** (metadata).

In [ ]:
# Pick a file from the list above. Some files were recorded through thick clouds and contain
# no usable shots at all (later cells then fail). Files that work: 2, 4, 5, 10, 12, 13, 14, 17, 19, 21, 25
FILE_NO = 2                                  # 2 = 5 June 2019: good shots, but also many cloud-affected ones to filter
gedi = h5py.File(files[FILE_NO], "r")        # open the file (read only)
list(gedi.keys())                            # the top-level groups

In [ ]:
# File-level metadata (stored as attributes)
meta = gedi["METADATA/DatasetIdentification"].attrs
for key in meta:
    print(key, ":", meta[key])

GEDI has 3 lasers. One laser is split into two **coverage beams**, and the other two lasers are **full power beams**.
Each of these four beams is dithered across-track to produce two ground tracks. This gives **8 beam tracks**
(`BEAM0000` … `BEAM1011`). Full power beams penetrate dense canopy better.

In [ ]:
beams = [name for name in gedi.keys() if name.startswith("BEAM")]

for b in beams:
    beam_type = gedi[b].attrs["description"]
    n_shots = len(gedi[b]["shot_number"])
    print(b, beam_type, "-", n_shots, "shots")

In [ ]:
# What is inside one beam group? Let's list the datasets of BEAM0101 (a full power beam).
for name in gedi["BEAM0101"].keys():
    print(name)

In [ ]:
# Each dataset has a description attribute that tells us what it contains
for name in ["rh", "elev_lowestmode", "elev_highestreturn", "quality_flag", "sensitivity", "degrade_flag"]:
    print(name, ":", gedi["BEAM0101"][name].attrs["description"], "\n")

## 3. From laser shots to a GeoDataFrame

Every GEDI **shot** is one laser footprint (~25 m diameter) on the ground. It has a unique `shot_number`.
`lat_lowestmode` and `lon_lowestmode` give the location of the lowest detected mode (the ground).

The function below reads the most important datasets of **one beam** into a table (one row per shot).
`rh` is a 2D array with one row per shot and 101 columns: the relative heights at 0, 1, 2 … 100 % of
returned energy. We keep only a few of them as columns (`rh[:, 98]` = column 98 = RH98).

In [ ]:
def read_beam(h5, beam):
    g = h5[beam]                     # the group of this beam
    rh = g["rh"][:]                  # all RH values: shape (shots, 101)
    table = pd.DataFrame({
        "beam": beam,
        "beam_type": g.attrs["description"],
        "idx": np.arange(len(rh)),   # row number inside the beam, to find the shot again later
        "shot_number": g["shot_number"][:].astype(str),
        "lat": g["lat_lowestmode"][:],
        "lon": g["lon_lowestmode"][:],
        "elev_ground": g["elev_lowestmode"][:],
        "elev_top": g["elev_highestreturn"][:],
        "dem_tandemx": g["digital_elevation_model"][:],
        "rh25": rh[:, 25],
        "rh50": rh[:, 50],
        "rh75": rh[:, 75],
        "rh98": rh[:, 98],
        "rh100": rh[:, 100],
        "quality_flag": g["quality_flag"][:],
        "degrade_flag": g["degrade_flag"][:],
        "sensitivity": g["sensitivity"][:],
        "selected_algorithm": g["selected_algorithm"][:],
    })
    return table

# Read all beams and stack the tables on top of each other
tables = []
for b in beams:
    tables.append(read_beam(gedi, b))
df = pd.concat(tables, ignore_index=True)

# Turn the table into a GeoDataFrame: every shot becomes a point (lon/lat, WGS84 = EPSG:4326)
points = gpd.points_from_xy(df.lon, df.lat)
gdf = gpd.GeoDataFrame(df, geometry=points, crs="EPSG:4326")

print(len(gdf), "shots in the region")
gdf.head()

In [ ]:
aoi = gpd.read_file(DATA / "aoi.geojson")       # our study area: Kalkalpen National Park

fig, ax = plt.subplots(figsize=(8, 6))
gdf.plot(ax=ax, column="beam", markersize=0.5, legend=True)
aoi.boundary.plot(ax=ax, color="red", lw=2)
ax.set(xlabel="Longitude", ylabel="Latitude", title="GEDI shots in the region and the study area (red)")
plt.show()

### 3.1 Clip to the study area

The file covers a whole region, but we only want the shots inside the study area.
`GeoDataFrame.clip()` keeps only the geometries inside a polygon.

> With full-size granules (1–3 GB) you would clip *before* reading everything: read only `lat_lowestmode` and
> `lon_lowestmode`, build a boolean mask, and then read only the masked rows of the large datasets (like `rh`).
> See `prep/prepare_subset.py` for an example.

In [ ]:
gdf_region = gdf                 # keep a copy of all shots
gdf = gdf_region.clip(aoi)       # keep only the shots inside the study area
print(len(gdf), "of", len(gdf_region), "shots are inside the study area")

# Interactive map: study area in red, shots coloured by beam
m = aoi.explore(style_kwds={"fill": False, "color": "red"})
gdf.explore(m=m, column="beam", tooltip=["shot_number", "beam_type", "quality_flag"],
            marker_kwds={"radius": 2})

Zoom in: the shots are arranged in parallel tracks, one per beam.

**Question:** Roughly how far apart are neighbouring shots along a track, and how far apart are the tracks?

## 4. Relative height metrics of a single shot

**How to read RH metrics:**
- `rh` gives the height (m) **above the ground** (`elev_lowestmode`) below which *x* % of the returned waveform energy lies.
- **RH100** = `elev_highestreturn` − `elev_lowestmode`, the top of the canopy as seen by the lidar.
- Low RH values (RH1–RH20) are often **negative**. Part of the ground return lies below the centre of the ground peak.
- RH metrics are meant for vegetated surfaces. Over bare ground or water they can be confusing.

First, pick a good-quality shot in tall forest (here: a shot close to the 90th percentile of RH98).

In [ ]:
# Keep only good-quality shots
good = gdf[(gdf.quality_flag == 1) & (gdf.degrade_flag == 0)]

# Pick the shot whose RH98 is closest to the 90th percentile (a tall forest shot)
target = good.rh98.quantile(0.9)
distance_to_target = (good.rh98 - target).abs()
shot = good.loc[distance_to_target.idxmin()]

shot[["shot_number", "beam", "idx", "lat", "lon", "elev_ground", "rh98"]]

In [ ]:
# Read all 101 RH values of this shot from the file
g = gedi[shot.beam]
rh_shot = g["rh"][shot.idx]                  # 101 values, metres above ground
z_ground = shot.elev_ground                  # ground elevation (m a.s.l.)
z_top = shot.elev_top                        # elevation of the highest return
rh_elev = rh_shot + z_ground                 # heights above ground -> elevations

percent = np.arange(101)                     # 0, 1, 2, ..., 100

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(percent, rh_elev, color="black", lw=2.5, label="RH profile")
ax.axhline(z_ground, color="saddlebrown", label="Ground (elev_lowestmode)")
ax.axhline(z_top, color="navy", label="RH100 (elev_highestreturn)")
ax.axhline(rh_elev[25], color="skyblue", ls="--", label="RH25")
ax.axhline(rh_elev[50], color="tab:blue", ls="--", label="RH50")
ax.axhline(rh_elev[75], color="tab:purple", ls="--", label="RH75")
ax.set(xlabel="Percent energy returned", ylabel="Elevation (m)", xlim=(0, 100),
       title=f"Shot {shot.shot_number}")
ax.legend(loc="lower right", fontsize=8)
plt.show()

**Questions:**
- How tall is the canopy at this shot?
- In which height range is most of the canopy material? (Where is the curve steepest?)

## 5. Beam transect and quality filtering

Let's look at one **full power beam** along its whole track inside our study area.

In [ ]:
# Find the full power beam with the most shots in the study area
full_power = gdf[gdf.beam_type == "Full power beam"]
beam = full_power.beam.value_counts().index[0]

# All shots of this beam, sorted from south to north
tr = gdf[gdf.beam == beam].sort_values("lat")

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(tr.lat, tr.elev_ground, s=3, color="saddlebrown", label="GEDI ground")
ax.scatter(tr.lat, tr.elev_top, s=3, color="green", label="GEDI canopy top")
ax.plot(tr.lat, tr.dem_tandemx, color="grey", lw=1, label="TanDEM-X DEM")
ax.set(xlabel="Latitude", ylabel="Elevation (m)", title=f"{beam}: all shots (unfiltered)")
ax.legend()
plt.show()

That looks messy: clouds and weak returns produce elevations far from the terrain. GEDI provides several
datasets for filtering:

| Dataset | Keep shots where | Why |
|---|---|---|
| `quality_flag` | `== 1` | shot meets criteria for energy, sensitivity, amplitude and surface tracking |
| `degrade_flag` | `== 0` | shot was not acquired during a period of degraded pointing/positioning |
| `sensitivity` | `> 0.95` | maximum canopy cover the shot can penetrate. `quality_flag` already uses 0.9 over land, but the GEDI user guide recommends higher thresholds for dense forest |

In [ ]:
def quality_filter(d, min_sensitivity=0.95):
    keep = (d.quality_flag == 1) & (d.degrade_flag == 0) & (d.sensitivity > min_sensitivity)
    return d[keep]

tr_f = quality_filter(tr)
print(len(tr_f), "of", len(tr), "shots kept")

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(tr_f.lat, tr_f.elev_ground, s=4, color="saddlebrown", label="GEDI ground")
ax.scatter(tr_f.lat, tr_f.elev_top, s=4, color="green", label="GEDI canopy top")
ax.plot(tr.lat, tr.dem_tandemx, color="grey", lw=1, label="TanDEM-X DEM")
ax.set(xlabel="Latitude", ylabel="Elevation (m)", title=f"{beam}: quality-filtered")
ax.legend()
plt.show()

### 5.1 The full RH profile along the transect

So far we only used RH98 as canopy top. Now we plot **all 101 RH values** of every shot along a ~6 km
stretch of the track. Each shot becomes a vertical column of green dots, from the ground (RH0) up to the
canopy top (RH100). The black line is the ground elevation.

To measure distances in metres, we first project the shots to UTM zone 33N (EPSG:32633).

In [ ]:
# Distance of each shot from the first shot of the track (in metres)
tr_utm = tr_f.to_crs("EPSG:32633")
first_shot = tr_utm.geometry.iloc[0]
tr_utm["dist_m"] = tr_utm.geometry.distance(first_shot)

# Choose a ~6 km stretch. Change START to move along the track!
START = 0          # metres
LENGTH = 6000      # metres
seg = tr_utm[(tr_utm.dist_m >= START) & (tr_utm.dist_m < START + LENGTH)]
print(len(seg), "shots in this stretch")

# Read all 101 RH values for these shots (one row per shot)
rh_all = gedi[beam]["rh"][:]
rh_seg = rh_all[seg.idx]                                      # shape (shots, 101)
rh_seg_elev = rh_seg + seg.elev_ground.values[:, np.newaxis]  # heights above ground -> elevations

fig, ax = plt.subplots(figsize=(12, 5))
for i in range(len(seg)):
    x = np.full(101, seg.dist_m.iloc[i])   # the same distance for all 101 RH values of one shot
    ax.scatter(x, rh_seg_elev[i], s=8, color="darkgreen", alpha=0.1)
ax.plot(seg.dist_m, seg.elev_ground, color="black", lw=2, label="Ground elevation")
ax.scatter([], [], color="darkgreen", alpha=0.3, label="RH 0-100")   # only for the legend
ax.set(xlabel="Distance along transect (m)", ylabel="Elevation (m)",
       title=f"Relative height metrics and ground elevation along {beam}")
ax.legend(loc="lower right")
plt.show()

**Questions:**
- Where along the stretch is the canopy tallest? Where is there (almost) no canopy?
- Where are the green dots dense, and where are they spread out? What does that tell you about the vertical structure of the forest?
- Change `START` (e.g. 6000, 12000) and look at another part of the track.

## 6. Map canopy height

In [ ]:
# Apply the quality filter to all shots in the study area
gdf_f = quality_filter(gdf)
print(len(gdf_f), "of", len(gdf), "shots kept")

m = aoi.explore(style_kwds={"fill": False, "color": "red"})
gdf_f.explore(m=m, column="rh98", cmap="viridis", vmin=0, vmax=40,
              tooltip=["shot_number", "beam", "rh98", "elev_ground", "sensitivity"],
              marker_kwds={"radius": 3})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].hist(gdf_f.rh98, bins=40)
axs[0].set(xlabel="RH98 (m)", ylabel="Shots", title="Canopy height distribution")

axs[1].scatter(gdf_f.elev_ground, gdf_f.rh98, s=4, alpha=0.5)
axs[1].set(xlabel="Ground elevation (m a.s.l.)", ylabel="RH98 (m)", title="Canopy height vs. elevation")

plt.tight_layout()
plt.show()

**Questions:**
- Why do we use RH98 instead of RH100 as "canopy height"?
- Some shots have RH98 above 50 m. Trees that tall are very unlikely in the Kalkalpen. Find these shots on the map.
  Where are they? (Hint: on steep slopes the waveform is stretched, because the ground under the footprint
  covers a large elevation range.)
- Why could some shots in the valleys have a very low RH98?

## 7. Export for GIS

In [ ]:
cols = ["shot_number", "beam", "beam_type", "elev_ground", "elev_top", "rh25", "rh50", "rh75",
        "rh98", "rh100", "sensitivity", "geometry"]
gdf_f[cols].to_file(OUT / "gedi_l2a_filtered.gpkg", driver="GPKG")
gdf_f[cols].to_file(OUT / "gedi_l2a_filtered.geojson", driver="GeoJSON")
print(list(OUT.iterdir()))

The files are in the `output/` folder (see the file browser on the left). Right-click a file and choose **Download**
to open it in QGIS.

## Exercises

1. Repeat the transect plot (Section 5) for a **coverage beam**. How many shots survive the quality filter compared to a full power beam?
2. Change the sensitivity threshold to 0.9 and 0.98. How do the number of shots and the canopy height distribution change?
3. Change `FILE_NO` in Section 2 to another file and run the notebook again. Do you see differences between the acquisitions (season, time of day: `solar_elevation`)?
4. Add `land_cover_data/landsat_treecover` to `read_beam()` and plot it against RH98.
5. Draw your own study area (e.g. on [geojson.io](https://geojson.io)), save it as GeoJSON in `data/`, and clip to it instead.

## Going further (optional)

### A. Different algorithm setting groups

GEDI processes every waveform with **six algorithm setting groups** (a1–a6). The values we used so far
come from the algorithm selected for that shot (`selected_algorithm`). The results of all six are stored in
the `geolocation` subgroup. Note: `rh_a<n>` is stored in **centimetres**.

This plot uses the single shot from Section 4.

In [ ]:
g = gedi[shot.beam]
fig, ax = plt.subplots(figsize=(5, 7))
for a in range(1, 7):
    rh_a = g[f"geolocation/rh_a{a}"][shot.idx] / 100          # cm -> m
    z_a = g[f"geolocation/elev_lowestmode_a{a}"][shot.idx]
    if a == shot.selected_algorithm:
        ax.plot(percent, rh_a + z_a, lw=3, label=f"a{a} (selected)")
    else:
        ax.plot(percent, rh_a + z_a, lw=1, label=f"a{a}")
ax.set(xlabel="Percent energy returned", ylabel="Elevation (m)", xlim=(0, 100),
       title="RH profile by algorithm setting group")
ax.legend(fontsize=8)
plt.show()

**Question:** Where do the algorithms differ most, near the ground or in the canopy? Why could that be?

### B. Extra check for mountain terrain

In steep terrain, GEDI sometimes picks the wrong mode as the ground. A simple extra check compares the GEDI
ground elevation with an independent DEM (TanDEM-X is included in the file) and drops shots with large differences.
The threshold is a choice. Try different values!

In [ ]:
gdf_dz = gdf_f.copy()
gdf_dz["dz"] = gdf_dz.elev_ground - gdf_dz.dem_tandemx       # GEDI ground minus DEM

plt.hist(gdf_dz.dz.clip(-100, 100), bins=80)
plt.xlabel("GEDI ground - TanDEM-X (m)")
plt.ylabel("Shots")
plt.show()

MAX_DZ = 50   # metres
gdf_dz = gdf_dz[gdf_dz.dz.abs() < MAX_DZ]
print(len(gdf_dz), "of", len(gdf_f), "quality-filtered shots pass the DEM check")

**Question:** Do the suspicious RH98 > 50 m shots from Section 6 survive this check?

In [ ]:
gedi.close()     # close the file when you are done

## Further reading
- GEDI L2A product page and user guide: https://doi.org/10.5067/GEDI/GEDI02_A.002
- LP DAAC GEDI tutorials (L1B waveforms, L2B canopy cover): https://github.com/nasa/GEDI-Data-Resources
- GEDI mission: https://gedi.umd.edu